In [6]:
import torch
import os
import sys
sys.path.append('/hdd/yang/projects/glomeruli_segmentation/2025-zhou-hipct-hierarchical-segmentation')
from segmentation.models.kidney_vnet.net.ResUnet import ResUNet

def load_model(checkpoint_path, model_name):
    checkpoint = torch.load(os.path.join(checkpoint_path, model_name))
    net = ResUNet(training=False, inchannel=1, stage = 1)
    if torch.cuda.is_available():
        net = torch.nn.DataParallel(net).cuda()
    net.load_state_dict(checkpoint['model_state_dict'])
    return net

In [7]:
# check model parameters - VNet
model_path = '/hdd/yang/projects/glomeruli_segmentation/data/publish_data/models/vnet/saves'
model = load_model(model_path, 'vnet_fold3_final.tar')
pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total number of trainable params: {pytorch_total_params}')

Total number of trainable params: 9324754


In [8]:
from torchinfo import summary
batch_size = 1
summary(model,
        input_size=(batch_size, 1, 128, 128, 128),   
        verbose=2,
        col_width=20,
        col_names=["kernel_size", "output_size", "num_params", "mult_adds"],
        row_settings=["var_names"],
        )

Layer (type (var_name))                  Kernel Shape         Output Shape         Param #              Mult-Adds
DataParallel (DataParallel)              --                   [1, 2, 128, 128, 128] --                   --
├─ResUNet (module)                       --                   [1, 2, 128, 128, 128] --                   --
│    └─encoder_stage1.0.weight           [16, 1, 3, 3, 3]                          ├─432
│    └─encoder_stage1.0.bias             [16]                                      ├─16
│    └─encoder_stage1.1.weight           [16]                                      ├─16
│    └─encoder_stage2.0.weight           [32, 32, 3, 3, 3]                         ├─27,648
│    └─encoder_stage2.0.bias             [32]                                      ├─32
│    └─encoder_stage2.1.weight           [32]                                      ├─32
│    └─encoder_stage2.2.weight           [32, 32, 3, 3, 3]                         ├─27,648
│    └─encoder_stage2.2.bias             [32]

Layer (type (var_name))                  Kernel Shape         Output Shape         Param #              Mult-Adds
DataParallel (DataParallel)              --                   [1, 2, 128, 128, 128] --                   --
├─ResUNet (module)                       --                   [1, 2, 128, 128, 128] --                   --
│    └─encoder_stage1.0.weight           [16, 1, 3, 3, 3]                          ├─432
│    └─encoder_stage1.0.bias             [16]                                      ├─16
│    └─encoder_stage1.1.weight           [16]                                      ├─16
│    └─encoder_stage2.0.weight           [32, 32, 3, 3, 3]                         ├─27,648
│    └─encoder_stage2.0.bias             [32]                                      ├─32
│    └─encoder_stage2.1.weight           [32]                                      ├─32
│    └─encoder_stage2.2.weight           [32, 32, 3, 3, 3]                         ├─27,648
│    └─encoder_stage2.2.bias             [32]

In [9]:
from monai.networks.nets import UNETR
def load_model(checkpoint_path, model_name):
    checkpoint = torch.load(os.path.join(checkpoint_path, model_name))
    net = UNETR(
        in_channels=1,
        out_channels=2,
        img_size=(128, 128, 128),
        feature_size=16,
        hidden_size=768,
        mlp_dim=3072,
        num_heads=12,
        proj_type="perceptron",
        norm_name="instance",
        res_block=True,
        dropout_rate=0.0
    )
    #if torch.cuda.is_available():
        #net = torch.nn.DataParallel(net).cuda()
    #net.load_state_dict(checkpoint['model_state_dict'])
    return net

In [10]:
model_path = '/hdd/yang/projects/glomeruli_segmentation/data/publish_data/models/unetr/saves/'
model = load_model(model_path, 'unetr_fold1_final.tar')
pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total number of trainable params: {pytorch_total_params}')

Total number of trainable params: 121350370


In [11]:
batch_size = 1
summary(model,
        input_size=(batch_size, 1, 128, 128, 128),   
        verbose=2,
        col_width=20,
        col_names=["kernel_size", "output_size", "num_params", "mult_adds"],
        row_settings=["var_names"],
        )

Layer (type (var_name))                                 Kernel Shape         Output Shape         Param #              Mult-Adds
UNETR (UNETR)                                           --                   [1, 2, 128, 128, 128] --                   --
├─ViT (vit)                                             --                   [1, 512, 768]        --                   --
│    └─patch_embedding.position_embeddings              [1, 512, 768]                             ├─393,216
│    └─patch_embedding.patch_embeddings.1.weight        [768, 4096]                               ├─3,145,728
│    └─patch_embedding.patch_embeddings.1.bias          [768]                                     ├─768
│    └─blocks.0.mlp.linear1.weight                      [3072, 768]                               ├─2,359,296
│    └─blocks.0.mlp.linear1.bias                        [3072]                                    ├─3,072
│    └─blocks.0.mlp.linear2.weight                      [768, 3072]                     

Layer (type (var_name))                                 Kernel Shape         Output Shape         Param #              Mult-Adds
UNETR (UNETR)                                           --                   [1, 2, 128, 128, 128] --                   --
├─ViT (vit)                                             --                   [1, 512, 768]        --                   --
│    └─patch_embedding.position_embeddings              [1, 512, 768]                             ├─393,216
│    └─patch_embedding.patch_embeddings.1.weight        [768, 4096]                               ├─3,145,728
│    └─patch_embedding.patch_embeddings.1.bias          [768]                                     ├─768
│    └─blocks.0.mlp.linear1.weight                      [3072, 768]                               ├─2,359,296
│    └─blocks.0.mlp.linear1.bias                        [3072]                                    ├─3,072
│    └─blocks.0.mlp.linear2.weight                      [768, 3072]                     

In [12]:
from monai.networks.nets import SwinUNETR
def load_model(checkpoint_path, model_name):
    checkpoint = torch.load(os.path.join(checkpoint_path, model_name))
    net = SwinUNETR(
        img_size=(128, 128, 128),
        in_channels=1,
        out_channels=2,
        depths=(2,2,2,2),
        num_heads=(3,6,12,24),
        feature_size=24,
        norm_name="instance"
    )
    return net

model_path = '/hdd/yang/projects/glomeruli_segmentation/data/publish_data/models/swinunetr/saves'
model = load_model(model_path, 'swin-unetr_fold0_final.tar')

In [13]:
batch_size = 1
summary(model,
        input_size=(batch_size, 1, 128, 128, 128),   
        verbose=2,
        col_width=20,
        col_names=["kernel_size", "output_size", "num_params", "mult_adds"],
        row_settings=["var_names"],
        )

Layer (type (var_name))                                 Kernel Shape         Output Shape         Param #              Mult-Adds
SwinUNETR (SwinUNETR)                                   --                   [1, 2, 128, 128, 128] --                   --
├─SwinTransformer (swinViT)                             --                   [1, 24, 64, 64, 64]  --                   --
│    └─patch_embed.proj.weight                          [24, 1, 2, 2, 2]                          ├─192
│    └─patch_embed.proj.bias                            [24]                                      ├─24
│    └─layers1.0.blocks.0.norm1.weight                  [24]                                      ├─24
│    └─layers1.0.blocks.0.norm1.bias                    [24]                                      ├─24
│    └─layers1.0.blocks.0.attn.relative_position_bias_table [2197, 3]                                 ├─6,591
│    └─layers1.0.blocks.0.attn.qkv.weight               [72, 24]                                  ├─1,7

Layer (type (var_name))                                 Kernel Shape         Output Shape         Param #              Mult-Adds
SwinUNETR (SwinUNETR)                                   --                   [1, 2, 128, 128, 128] --                   --
├─SwinTransformer (swinViT)                             --                   [1, 24, 64, 64, 64]  --                   --
│    └─patch_embed.proj.weight                          [24, 1, 2, 2, 2]                          ├─192
│    └─patch_embed.proj.bias                            [24]                                      ├─24
│    └─layers1.0.blocks.0.norm1.weight                  [24]                                      ├─24
│    └─layers1.0.blocks.0.norm1.bias                    [24]                                      ├─24
│    └─layers1.0.blocks.0.attn.relative_position_bias_table [2197, 3]                                 ├─6,591
│    └─layers1.0.blocks.0.attn.qkv.weight               [72, 24]                                  ├─1,7

In [8]:
import skimage.io as skio
import numpy as np
import glob
import os 

label_dir = '/hdd/yang/projects/glomeruli_segmentation/2025-zhou-hipct-hierarchical-segmentation/data/nnUNet_preprocessed/Dataset009_25-08Glom_search_w_fat_label_filtered/gt_segmentations/'
label_list = sorted(glob.glob(os.path.join(label_dir, '*.tif')))
label_proportion = []
for label_path in label_list:
    label = skio.imread(label_path)
    total_voxels = np.prod(label.shape)
    glom_voxels = np.sum(label > 0)
    proportion = glom_voxels / total_voxels
    label_proportion.append(proportion)
label_proportion = np.array(label_proportion)
print(f'Number of label files: {len(label_list)}')
print(f'Average glomeruli voxel proportion: {np.mean(label_proportion)}')
print(f'Standard deviation of glomeruli voxel proportion: {np.std(label_proportion)}')
print(f'Minimum glomeruli voxel proportion: {np.min(label_proportion)}')

Number of label files: 1317
Average glomeruli voxel proportion: 0.026277265860080355
Standard deviation of glomeruli voxel proportion: 0.010173253191254604
Minimum glomeruli voxel proportion: 0.0101318359375
